In [1]:
# make sure data files are loadable
import pickle
from pathlib import Path 

DATA_PROCESSED = Path("../data/processed")

required = ["train.pkl", "test.pkl", "train_demographics.pkl", "test_demographics.pkl"]
for fn in required:
    p = DATA_PROCESSED / fn
    print(fn, "exists:", p.exists(), "size_MB:", round(p.stat().st_size/1024/1024,1) if p.exists() else None)

def load_pkl(name):
    with open(DATA_PROCESSED / name, "rb") as f:
        return pickle.load(f)
    
train_df = load_pkl("train.pkl")
test_df = load_pkl("test.pkl")
train_demo_df = load_pkl("train_demographics.pkl")
test_demo_df = load_pkl("test_demographics.pkl")

print("train_df:", train_df.shape)
print("test_df:", test_df.shape)
print("train_demo_df:", train_demo_df.shape)
print("test_demo_df:", test_demo_df.shape)

train.pkl exists: True size_MB: 1491.1
test.pkl exists: True size_MB: 0.3
train_demographics.pkl exists: True size_MB: 0.0
test_demographics.pkl exists: True size_MB: 0.0
train_df: (574945, 341)
test_df: (107, 336)
train_demo_df: (81, 8)
test_demo_df: (2, 8)


In [2]:
# find out the columns -- label, subject_id, IMU/thermo/TOF
print("train_df columns:", len(train_df.columns))
print(train_df.columns[:30])
print(train_df.columns[-30:])

print("\ntrain_demo columns:", train_demo_df.columns.tolist())

train_df columns: 341
Index(['row_id', 'sequence_type', 'sequence_id', 'sequence_counter', 'subject',
       'orientation', 'behavior', 'phase', 'gesture', 'acc_x', 'acc_y',
       'acc_z', 'rot_w', 'rot_x', 'rot_y', 'rot_z', 'thm_1', 'thm_2', 'thm_3',
       'thm_4', 'thm_5', 'tof_1_v0', 'tof_1_v1', 'tof_1_v2', 'tof_1_v3',
       'tof_1_v4', 'tof_1_v5', 'tof_1_v6', 'tof_1_v7', 'tof_1_v8'],
      dtype='str')
Index(['tof_5_v34', 'tof_5_v35', 'tof_5_v36', 'tof_5_v37', 'tof_5_v38',
       'tof_5_v39', 'tof_5_v40', 'tof_5_v41', 'tof_5_v42', 'tof_5_v43',
       'tof_5_v44', 'tof_5_v45', 'tof_5_v46', 'tof_5_v47', 'tof_5_v48',
       'tof_5_v49', 'tof_5_v50', 'tof_5_v51', 'tof_5_v52', 'tof_5_v53',
       'tof_5_v54', 'tof_5_v55', 'tof_5_v56', 'tof_5_v57', 'tof_5_v58',
       'tof_5_v59', 'tof_5_v60', 'tof_5_v61', 'tof_5_v62', 'tof_5_v63'],
      dtype='str')

train_demo columns: ['subject', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']


In [3]:
#missing values in train_df
na_top = train_df.isna().sum().sort_values(ascending=False).head(20)
print(na_top)

# find possible column groups by checking column name patterns
candidate_labels =  [c for c in train_df.columns if "label" in c.lower() or "target" in c.lower() or "y" == c.lower()]

print("candidate labels cols:", candidate_labels)

label_col = None
if len(candidate_labels) == 1:
    label_col = candidate_labels[0]
elif len(candidate_labels) > 1:
    # pick the first one for now, we can adjust after seeing values
    label_col = candidate_labels[0]

print("label_col:", label_col)

if label_col:
    print(train_df[label_col].value_counts(dropna=False).head(30))
    

thm_5        33286
tof_5_v63    30142
tof_5_v24    30142
tof_5_v19    30142
tof_5_v20    30142
tof_5_v21    30142
tof_5_v22    30142
tof_5_v23    30142
tof_5_v25    30142
tof_5_v17    30142
tof_5_v26    30142
tof_5_v27    30142
tof_5_v28    30142
tof_5_v29    30142
tof_5_v30    30142
tof_5_v31    30142
tof_5_v18    30142
tof_5_v16    30142
tof_5_v33    30142
tof_5_v6     30142
dtype: int64
candidate labels cols: []
label_col: None


In [4]:
# find idlike columns for grouping
id_cols = [c for c in train_df.columns if c.lower().endswith("_id") or "subject" in c.lower() or "series" in c.lower() or "sequence" in c.lower()]
print("id-like columns:", id_cols)

# modality groups
imu_cols = [c for c in train_df.columns if any(k in c.lower() for k in ["imu","acc","gyro","mag"])]
thermo_cols = [c for c in train_df.columns if any(k in c.lower() for k in ["thermo","temp","thermal"])]
tof_cols = [c for c in train_df.columns if "tof" in c.lower()]

print("IMU cols:", len(imu_cols))
print("Thermo cols:", len(thermo_cols))
print("TOF cols:", len(tof_cols))

id-like columns: ['row_id', 'sequence_type', 'sequence_id', 'sequence_counter', 'subject']
IMU cols: 3
Thermo cols: 0
TOF cols: 320


In [5]:
# Check processed train format (DataFrame vs dict/arrays) before EDA
print(type(train_df))
try:
    print(train_df.head(2))
except Exception as e:
    print("cannot head:", e)
    

<class 'pandas.DataFrame'>
              row_id sequence_type sequence_id  sequence_counter      subject  \
0  SEQ_000007_000000        Target  SEQ_000007                 0  SUBJ_059520   
1  SEQ_000007_000001        Target  SEQ_000007                 1  SUBJ_059520   

                       orientation                                   behavior  \
0  Seated Lean Non Dom - FACE DOWN  Relaxes and moves hand to target location   
1  Seated Lean Non Dom - FACE DOWN  Relaxes and moves hand to target location   

        phase             gesture     acc_x  ...  tof_5_v54  tof_5_v55  \
0  Transition  Cheek - pinch skin  6.683594  ...       -1.0       -1.0   
1  Transition  Cheek - pinch skin  6.949219  ...       -1.0       -1.0   

   tof_5_v56  tof_5_v57  tof_5_v58  tof_5_v59  tof_5_v60  tof_5_v61  \
0       -1.0       -1.0       -1.0       -1.0       -1.0       -1.0   
1       -1.0       -1.0       -1.0       -1.0       -1.0       -1.0   

   tof_5_v62  tof_5_v63  
0       -1.0       -1.